# 00 Quality control of the observed and simulated flow

Before reading any score, this notebook checks the two data sets behind it, gage by gage:

1. **CFE model runs** (AORC, MRMS Pass 2, MRMS radar-only): zero flow, missing or negative values, runs that do not differ, runoff depth that is not plausible for the drainage area.
2. **USGS observations**: coverage of the scoring window, long gaps, dry streams, flat lines.
3. **Screening rules**: each rule flags gages, and flagged gages are removed. The list of kept gages is saved in `results/tables/qaqc_sites.csv`; notebooks 01 and 02 use only those gages.

All statistics are for the scoring window, 1 Oct 2021 to 31 Dec 2025 (UTC).

**Paths.** Nothing to edit. The notebook finds the repository by itself and takes the data from the folder next to it (`cfe_evel/organized` and `cfe_evel/results`), so it runs the same from IDAS (`~/hpchome/Papers/cfe_evel/cfe_eval/analysis`) and from Argon. To use another data folder, set the environment variable `CFE_DATA_ROOT` before starting Jupyter.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# find the repository (the folder that holds cfe_eval/config.py), whatever folder the notebook is opened from
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cfe_eval" / "config.py").exists())
sys.path.insert(0, str(REPO))
from cfe_eval import common as cm
from cfe_eval import config as cfg
from cfe_eval import maps, plots

%matplotlib inline
plots.slide_style()
pd.set_option("display.max_columns", 60, "display.width", 220)

FIG = cfg.FIGURES / "00_qaqc"            # figures of this notebook
FIG.mkdir(parents=True, exist_ok=True)
SAVE = True                             # write every figure as PNG, 300 dpi

def save(fig, name):
    if SAVE:
        fig.savefig(FIG / f"{name}.png")

print("repository:", REPO)
print("data root :", cfg.DATA_ROOT)
print("tables    :", cfg.TABLES, "(found)" if cfg.TABLES.exists() else "(MISSING: run the evaluation jobs first)")
print("figures   :", FIG)

from cfe_eval import qaqc

## 1. Quality statistics per gage

The first run reads the 818 hourly files (one to two minutes) and stores the result in `results/tables/qaqc_stats`. Later runs load that table. Set `RECOMPUTE = True` after changing the organized data.

In [ ]:
RECOMPUTE = False
stats = qaqc.compute_stats(force=RECOMPUTE)
gages = cm.read_table("gages")
P = list(cfg.PRODUCTS)
print(len(stats), "gages,", len(stats.columns), "columns")
stats.head(3)

## 2. Zero flow in the CFE runs

Question: does the model give exactly zero flow, where, and how often?

Two definitions are counted: exactly zero, and near zero (below 0.0001 m3/s, that is 0.1 litre per second).

In [ ]:
rows = []
for p in P:
    z, nz = stats[f"sim_zero_frac_{p}"], stats[f"sim_near_zero_frac_{p}"]
    rows.append({"product": cfg.PRODUCT_LABEL[p],
                 "gages with any zero hour": int((z > 0).sum()),
                 "zero more than 1 % of hours": int((z > 0.01).sum()),
                 "zero more than 10 %": int((z > 0.10).sum()),
                 "zero more than 50 %": int((z > 0.50).sum()),
                 "largest share of zero hours (%)": round(100 * z.max(), 1),
                 "longest zero spell (days)": round(stats[f"sim_longest_zero_days_{p}"].max(), 1),
                 "near zero more than 10 %": int((nz > 0.10).sum())})
zero_table = pd.DataFrame(rows).set_index("product")
zero_table.to_csv(FIG / "zero_flow_summary.csv")
zero_table

In [ ]:
# by climate type: number of gages with zero flow more than 1 % of the hours
by_zone = pd.DataFrame({cfg.PRODUCT_LABEL[p]: (stats[f"sim_zero_frac_{p}"] > 0.01).groupby(stats["aridity_zone"]).sum()
                        for p in P}).reindex(cfg.ZONES)
by_zone["USGS observed"] = (stats["obs_zero_frac"] > 0.01).groupby(stats["aridity_zone"]).sum()
by_zone["gages"] = stats.groupby("aridity_zone").size()
by_zone

In [ ]:
# is the model dry where the stream is dry? one point per gage
fig, axes = plt.subplots(1, 3, figsize=(20, 7), sharey=True)
zc = {"Arid": "#b5651d", "Semi-arid": "#e0a030", "Semi-humid": "#6aa84f", "Humid": "#1f5fa8"}
for ax, p in zip(axes, P):
    for z in cfg.ZONES:
        d = stats[stats["aridity_zone"] == z]
        ax.scatter(100 * d["obs_zero_frac"], 100 * d[f"sim_zero_frac_{p}"], s=55, color=zc[z],
                   edgecolor="black", linewidth=0.4, label=z)
    ax.plot([0, 100], [0, 100], color="black", lw=1)
    ax.set_title(cfg.PRODUCT_LABEL[p])
    ax.set_xlabel("Observed hours with zero flow (%)")
    ax.grid(color="0.9")
axes[0].set_ylabel("Simulated hours with zero flow (%)")
fig.tight_layout(rect=(0, 0, 1, 0.93))
h, lab = axes[0].get_legend_handles_labels()
fig.legend(h, lab, loc="upper center", ncol=4, frameon=True, edgecolor="black", fancybox=False)
save(fig, "zero_flow_sim_vs_obs")

Reading the figure: points on the bottom edge are streams that go dry in reality while the model keeps some flow; points on the left edge are streams that never go dry while the model does.

The gages with the most zero flow in the model:

In [ ]:
cols = ["gage", "state_tag", "aridity_zone", "area_km2", "obs_zero_frac"] + [f"sim_zero_frac_{p}" for p in P] \
       + [f"sim_longest_zero_days_{p}" for p in P]
stats.sort_values("sim_zero_frac_AORC", ascending=False)[cols].head(15).round(3)

In [ ]:
def show_gage(site, start=None, end=None, log=False):
    """Observed and simulated hourly flow of one gage (scoring window by default)."""
    df = cm.read_timeseries(site).loc[start:end]
    g = gages.set_index("site").loc[site]
    fig, ax = plt.subplots(figsize=(13.33, 5.5))
    ax.plot(df.index, df["obs_scoring_cms"], color="black", lw=1.6, label="USGS observed")
    for p in P:
        ax.plot(df.index, df[p], color=plots.PRODUCT_COLORS[p][0], lw=1.1, label=cfg.PRODUCT_LABEL[p])
    if log:
        ax.set_yscale("log")
    ax.set_ylabel("Flow (m3/s)")
    ax.set_title(f"{site}  {g['name']}  ({g['area_km2']:.0f} km2, {g['aridity_zone']})", fontsize=15)
    ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    ax.grid(color="0.9")
    fig.tight_layout()
    return fig

worst = stats.sort_values("sim_zero_frac_AORC", ascending=False)["gage"].iloc[0]
fig = show_gage(worst)
save(fig, f"example_zero_flow_{worst}")

`show_gage("01234567", "2023-06-01", "2023-09-30", log=True)` plots any gage and period. It is the quickest way to look at a gage that a rule flags.

## 3. Other checks of the model runs

In [ ]:
def r(x, n):
    return round(float(x), n)

checks = pd.DataFrame({
    "gages with missing values": [int((stats[f"sim_missing_frac_{p}"] > 0).sum()) for p in P],
    "gages with negative flow": [int((stats[f"sim_negative_hours_{p}"] > 0).sum()) for p in P],
    "gages with constant flow": [int(stats[f"sim_constant_{p}"].sum()) for p in P],
    "median runoff depth (mm per year)": [r(stats[f"sim_runoff_mm_yr_{p}"].median(), 0) for p in P],
    "largest runoff depth (mm per year)": [r(stats[f"sim_runoff_mm_yr_{p}"].max(), 0) for p in P],
    "largest unit peak (m3/s per km2)": [r(stats[f"sim_unit_peak_cms_km2_{p}"].max(), 1) for p in P],
}, index=[cfg.PRODUCT_LABEL[p] for p in P]).T
checks.loc["median runoff depth (mm per year)", "USGS observed"] = r(stats["obs_runoff_mm_yr"].median(), 0)
checks.loc["largest runoff depth (mm per year)", "USGS observed"] = r(stats["obs_runoff_mm_yr"].max(), 0)
checks.loc["largest unit peak (m3/s per km2)", "USGS observed"] = r(stats["obs_unit_peak_cms_km2"].max(), 1)
checks

**Do the three runs differ?** If the rain had not been replaced in an MRMS run, its flow would equal the AORC flow hour by hour. The table gives the share of hours with exactly the same flow in two runs. A large share is expected only where both runs are at zero flow.

In [ ]:
ident = stats.filter(like="identical_frac_")
pd.DataFrame({"median share of identical hours (%)": (100 * ident.median()).round(3),
              "largest share (%)": (100 * ident.max()).round(1),
              "gages above 99 %": (ident > 0.99).sum()})

**Runoff depth.** Mean flow divided by the USGS drainage area, as a depth per year. It cannot exceed the yearly rain. A very large simulated depth means the model outlet collects water from a larger area than the gage; a very large observed depth means the area is wrong or water enters the basin another way. Gages beyond the dotted lines (2000 mm per year) are labelled.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
x, y = stats["obs_runoff_mm_yr"], stats["sim_runoff_mm_yr_AORC"]
for z in cfg.ZONES:
    m = stats["aridity_zone"] == z
    ax.scatter(x[m], y[m], s=45, color=zc[z], edgecolor="black", linewidth=0.4, label=z)
lim = (1, 30000)
ax.plot(lim, lim, color="black", lw=1)
for k in (0.1, 10):
    ax.plot(lim, [k * v for v in lim], color="0.5", lw=1, linestyle=(0, (5, 4)))
ax.axvline(2000, color="0.3", lw=1, linestyle=":")
ax.axhline(2000, color="0.3", lw=1, linestyle=":")
for _, r in stats[(x > 2000) | (y > 2000)].iterrows():
    ax.annotate(r["gage"], (r["obs_runoff_mm_yr"], r["sim_runoff_mm_yr_AORC"]), fontsize=11,
                xytext=(5, 5), textcoords="offset points")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Observed runoff depth (mm per year)")
ax.set_ylabel("Simulated runoff depth, AORC (mm per year)")
ax.legend(frameon=True, edgecolor="black", fancybox=False, loc="lower right")
ax.grid(color="0.9")
fig.tight_layout()
save(fig, "runoff_depth_sim_vs_obs")

Solid line: simulated = observed. Dashed lines: ten times more and ten times less. Points far from the solid line with every rain product are not a rain problem; rule `volume_mismatch` flags them.

## 4. Checks of the USGS observations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].hist(100 * stats["obs_coverage"], bins=np.arange(0, 102.5, 2.5), color="0.35", edgecolor="white")
axes[0].set_xlabel("Hours with observed flow (%)")
axes[0].set_ylabel("Number of gages")
axes[0].set_yscale("log")
axes[1].hist(stats["obs_longest_gap_days"].clip(upper=730), bins=np.arange(0, 760, 30), color="0.35", edgecolor="white")
axes[1].set_xlabel("Longest gap (days, capped at 730)")
axes[1].set_yscale("log")
axes[2].hist(100 * stats["obs_flat_frac"], bins=np.arange(0, 31, 1), color="0.35", edgecolor="white")
axes[2].set_xlabel("Hours in flat lines of 5 days or more (%)")
axes[2].set_yscale("log")
for ax in axes:
    ax.grid(axis="y", color="0.9")
    ax.set_axisbelow(True)
fig.tight_layout()
save(fig, "usgs_coverage_gaps_flat_lines")

obs_summary = pd.DataFrame({
    "coverage below 50 %": [(stats["obs_coverage"] < 0.50).sum()],
    "coverage below 75 %": [(stats["obs_coverage"] < 0.75).sum()],
    "coverage below 90 %": [(stats["obs_coverage"] < 0.90).sum()],
    "a gap longer than 1 year": [(stats["obs_longest_gap_days"] > 365).sum()],
    "any zero flow": [(stats["obs_zero_frac"] > 0).sum()],
    "dry more than half the time": [(stats["obs_zero_frac"] > 0.5).sum()],
    "flat lines above 5 % of hours": [(stats["obs_flat_frac"] > 0.05).sum()],
}, index=["number of gages"]).T
obs_summary

**Flat lines.** A natural stream does not hold exactly the same flow for five days. Long flat lines point to regulated releases below a dam, ice, or estimated record. The gage with the largest share:

In [ ]:
flat = stats.sort_values("obs_flat_frac", ascending=False)[["gage", "state_tag", "aridity_zone", "area_km2",
                                                             "obs_flat_frac", "obs_zero_frac", "obs_coverage"]]
display(flat.head(15).round(3))
fig = show_gage(flat["gage"].iloc[0], "2023-01-01", "2023-12-31")
save(fig, f"example_flat_lines_{flat['gage'].iloc[0]}")

## 5. Screening rules and the list of gages to keep

Each rule has a threshold and a switch. `drop: True` removes the flagged gages from the analysis; `drop: False` only marks them (column `flags_only`) and keeps them in. Change any value below and rerun from here.

**By default every rule drops its flagged gages**, so the other notebooks work only with gages that passed all checks. Keep in mind what this costs: three of the rules (`obs_mostly_dry`, `volume_mismatch`, `few_observed_events`) mostly remove gages in the dry climate types, where the cause may be the model or the climate and not the data. The table below the rules shows how many gages each rule removes in each climate type.

In [ ]:
import copy
RULES = copy.deepcopy(qaqc.DEFAULT_RULES)

# examples of changes
# RULES["obs_coverage_low"]["min_coverage"] = 0.90
# RULES["obs_mostly_dry"]["drop"] = False        # keep the ephemeral streams (flag only)
# RULES["volume_mismatch"]["drop"] = False       # keep the gages with a large volume mismatch (flag only)

rules_table = pd.DataFrame([{"rule": k, "drop": v["drop"],
                             "threshold": ", ".join(f"{a} = {b}" for a, b in v.items() if a not in ("drop", "why")),
                             "why": v["why"]} for k, v in RULES.items()]).set_index("rule")
with pd.option_context("display.max_colwidth", None):
    display(rules_table)

In [ ]:
flags = qaqc.apply_rules(stats, gages, RULES)
flags = flags.merge(stats[["gage", "aridity_zone", "area_class", "state_tag"]], on="gage")
rule_names = list(RULES)

per_rule = flags.groupby("aridity_zone")[rule_names].sum().reindex(cfg.ZONES).T
per_rule["all"] = per_rule.sum(axis=1)
per_rule.insert(0, "drop", [RULES[k]["drop"] for k in per_rule.index])
print("gages flagged by each rule, by climate type")
display(per_rule)

kept = pd.DataFrame({"gages": flags.groupby("aridity_zone").size(),
                     "kept": flags.groupby("aridity_zone")["keep"].sum()}).reindex(cfg.ZONES)
kept.loc["all"] = kept.sum()
kept["dropped"] = kept["gages"] - kept["kept"]
kept["kept (%)"] = (100 * kept["kept"] / kept["gages"]).round(1)
print("result of the screening")
kept

In [ ]:
out = flags.merge(stats.drop(columns=["aridity_zone", "area_class", "state_tag", "area_km2"]), on="gage")
out = out.merge(gages[["site", "name", "lat", "lon", "area_km2"]].rename(columns={"site": "gage"}), on="gage")
out.to_csv(cfg.TABLES / "qaqc_sites.csv", index=False)
print("written:", cfg.TABLES / "qaqc_sites.csv", f"({int(out['keep'].sum())} gages kept of {len(out)})")

with pd.option_context("display.max_rows", 200):
    display(out.loc[~out["keep"], ["gage", "name", "state_tag", "aridity_zone", "area_km2", "drop_reasons",
                                   "obs_coverage", "obs_flat_frac", "obs_runoff_mm_yr", "sim_runoff_mm_yr_AORC"]]
            .sort_values(["drop_reasons", "gage"]).round(2))

In [ ]:
status = np.where(out["keep"], "Kept", "Dropped")
fig = maps.figure_category_map(out, status, {"Kept": "#4d4d4d", "Dropped": "#e31a1c"},
                               title="Gages kept and dropped by the quality control")
save(fig, "map_kept_dropped")

The map uses the Albers Equal Area projection for the conterminous United States (EPSG:5070); the background shows the aridity zones.

## 6. Does the screening change the picture?

Medians over gages of the full-period scores, before and after dropping the screened gages.

In [ ]:
cont = cm.read_table("continuous_metrics")
full = cont[cont["period_type"] == "full"]
keep = set(out.loc[out["keep"], "gage"])
cols = ["NNSE", "KGE", "PBIAS_pct", "NRMSE_pct"]
before = full.groupby("product")[cols].median()
after = full[full["gage"].isin(keep)].groupby("product")[cols].median()
pd.concat({"all 818 gages": before, f"{len(keep)} kept gages": after}, axis=1).reindex(P).round(3)

## 7. Final list of gages

The gages below passed every rule with `drop: True`. From here on, notebooks 01 and 02 use only these gages: they read `results/tables/qaqc_sites.csv` and stop with a message if it is missing. A plain list of the kept gages is also written to `results/tables/qaqc_kept_gages.csv`.

In [ ]:
final = out.loc[out["keep"], ["gage", "name", "state_tag", "lat", "lon", "area_km2", "area_class", "aridity_zone", "flags_only"]]
final.to_csv(cfg.TABLES / "qaqc_kept_gages.csv", index=False)
summary = pd.DataFrame({"gages before": out.groupby("aridity_zone").size(),
                        "gages kept": final.groupby("aridity_zone").size()}).reindex(cfg.ZONES).fillna(0).astype(int)
summary.loc["all"] = summary.sum()
summary["removed"] = summary["gages before"] - summary["gages kept"]
print(f"{len(final)} gages kept for the analysis, {len(out) - len(final)} removed")
print("written:", cfg.TABLES / "qaqc_kept_gages.csv")
summary

## Ideas to go further

- Regulation: the package has no dam or land-use information. Joining GAGES-II (reference or non-reference class, dam storage) or the USGS peak-flow qualification codes would give a direct flag.
- Drainage area: comparing the USGS area with the area of the NextGen hydrofabric subset would confirm the outlet mismatches flagged by the runoff depth.
- Dry streams: for gages flagged `obs_mostly_dry`, scores normalized by the mean flow (NRMSE, PB) are unstable; event detection may be the better evaluation there.
- Observation quality codes (approved, provisional, estimated, ice) are not in the package; they could replace the flat-line rule.